# Lesson 13: Retrieval-augmented generation (RAG)

*Companion notebook for [the lesson page](../lessons/13-rag.html). Worked solutions are in `solutions/13-rag-solutions.ipynb`; try each exercise first.*

Make a model answer from your own documents, with citations, by building the full pipeline without a framework (parsing, chunking, hybrid retrieval, reranking and grounded generation), then measure it and fix what it gets wrong.

**Runs on:** Any computer (CPU). Retrieval downloads two small models; generating answers needs an API key.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy anthropic python-dotenv sentence-transformers

In [ ]:
import numpy as np
from dotenv import load_dotenv
load_dotenv()   # ANTHROPIC_API_KEY for the generation step

## Ingestion: parsing and chunking

### Chunking

- **Too small** (a sentence): precise matches, but the chunk lacks the context needed to answer ("Set it to 5" — set what?).
- **Too large** (a whole chapter): the embedding becomes a blurry average of many topics, retrieval gets less precise, and you waste context tokens.
- A common starting point is **200 to 800 tokens**, split at natural boundaries (sections, then paragraphs, then sentences), with a little **overlap** so ideas spanning a boundary survive.

In [ ]:
import re

def split_long(paragraph: str, max_chars: int) -> list[str]:
    """Split an over-long paragraph at sentence boundaries."""
    if len(paragraph) <= max_chars:
        return [paragraph]
    out, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+", paragraph):
        if cur and len(cur) + len(s) + 1 > max_chars:
            out.append(cur)
            cur = s
        else:
            cur = f"{cur} {s}".strip()
    return out + [cur] if cur else out

def chunk_text(text: str, max_chars: int = 2000, overlap_chars: int = 200) -> list[str]:
    """Pack paragraphs into chunks of about max_chars, carrying a little overlap forward."""
    paras = [piece for p in re.split(r"\n\s*\n", text) if p.strip()
             for piece in split_long(p.strip(), max_chars)]
    chunks, cur = [], ""
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            tail = cur[-overlap_chars:] if overlap_chars else ""
            cur = tail[tail.find(" ") + 1:] if " " in tail else tail   # start the overlap at a word boundary
        cur = f"{cur}\n\n{p}" if cur else p
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# A small, made-up document collection about a field photometer, so this notebook runs end to end.
# Replace it with your own documents for the hands-on exercise.
SAMPLE_DOCS = [
    {"title": "Desiccant and humidity", "text": """Desiccant and humidity

The enclosure contains a desiccant cartridge that keeps the optics dry. Replace the cartridge every six months, or every three months at coastal and other humid sites (relative humidity above 70% for most of the year).

The indicator window turns from blue to pink when the desiccant is saturated. A saturated cartridge causes condensation on the entrance window, which shows up as a slow drop in measured signal over several days."""},
    {"title": "Filter wheel", "text": """Filter wheel

The filter wheel holds six filters and is driven by a stepper motor through a toothed belt. Error E-4071 means the wheel did not reach the commanded position within two seconds.

The most common cause of E-4071 is a slipping or worn belt. Replace the belt (part FW-BELT-02) and tighten the motor mount screws to 0.8 N·m. If E-4071 persists after a belt replacement, check the home sensor cable for damage."""},
    {"title": "Sun tracker and GPS", "text": """Sun tracker and GPS

The sun tracker points the instrument using the GPS position and time. Error E-2210 means the GPS has no fix; the tracker then falls back to the internal clock, which drifts by about two seconds per day.

If the tracker loses the sun, first check that the GPS antenna has a clear view of the sky and that the head is level to within 0.5 degrees, then run the sun-search routine from the service menu."""},
    {"title": "Data upload", "text": """Data upload

Measurements are uploaded nightly at 01:00 UTC over the cellular modem. Failed uploads are retried every 30 minutes for six hours. Data is buffered locally for up to 30 days, so a delayed upload does not lose measurements.

Firmware 2.3 has a known issue where the modem can stall after a failed upload, stopping all logging until a reboot. Firmware 2.4 fixes this."""},
    {"title": "Calibration", "text": """Calibration

Radiometric calibration must be checked every twelve months against the reference lamp, and after any change to the optics, filter wheel or detector. Record the lamp serial number and the calibration factors in the instrument log.

A change of more than 2% in any calibration factor since the previous check must be reported to the calibration team before the data is used."""},
    {"title": "Detector temperature and noise", "text": """Detector temperature and noise

The detector is cooled to 15 °C by a thermoelectric cooler. Dark current roughly doubles for every 7 °C rise in detector temperature, so a failed cooler shows up as higher noise, especially at short wavelengths.

If the detector temperature reading exceeds 25 °C, check the cooler fan and clean dust from the heat sink."""},
    {"title": "Power and reboot", "text": """Power and reboot

The instrument runs from 24 V DC. A power cycle resets the tracker and the filter wheel, and logging resumes within five minutes. Frequent power cycles shorten the life of the filter wheel motor; more than 40 cycles per month should be investigated."""},
    {"title": "Support requests", "text": """Support requests

Send support requests to the service desk with the instrument id, the firmware version and a description of the problem. Attach the last 24 hours of the system log."""},
]
print(len(SAMPLE_DOCS), "documents")

In [ ]:
for d in SAMPLE_DOCS[:2]:
    for c in chunk_text(d["text"], max_chars=300, overlap_chars=60):
        print(repr(c[:70]), "...", len(c), "chars")

> **Give every chunk its context**
>
> A chunk saying "Tighten to 2 N·m" is useless if retrieval cannot tell it belongs to "Section 4.2: Replacing the filter wheel belt" of the model X manual. Prepend the document title and section path to each chunk's text before embedding. Going further, **contextual retrieval** asks an LLM to write one or two sentences situating each chunk within its whole document, and prepends that. Anthropic reported this substantially reduced retrieval failures, and it combines well with prompt caching to keep the cost down.

## Retrieval

### Keyword retrieval with BM25

BM25 scores documents by how many query terms they contain, weighted by how rare each term is across the collection (rare terms such as error codes count for a lot), with diminishing returns for repeats and a correction for document length. It is decades old and still excellent for exact terms. Here it is from scratch:

In [ ]:
import math
from collections import Counter

def tokenize(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())     # keeps codes like "e-4071" whole

class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.toks = [tokenize(d) for d in docs]
        self.avgdl = sum(map(len, self.toks)) / len(docs)
        df = Counter(w for t in self.toks for w in set(t))           # in how many docs each word appears
        N = len(docs)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
        self.tf = [Counter(t) for t in self.toks]

    def scores(self, query: str) -> list[float]:
        q = tokenize(query)
        out = []
        for tf, toks in zip(self.tf, self.toks):
            norm = self.k1 * (1 - self.b + self.b * len(toks) / self.avgdl)
            out.append(sum(self.idf[w] * tf[w] * (self.k1 + 1) / (tf[w] + norm) for w in q if w in tf))
        return out

docs = ["Error E-4071 indicates a filter wheel position fault.",
        "Filter wheel faults are often caused by a slipping belt.",
        "Error E-2210 means the GPS has no fix.",
        "Replace the desiccant every six months."]
print([round(s, 2) for s in BM25(docs).scores("what does error E-4071 mean")])
# [1.9, 0.0, 0.69, 0.0]: the exact code dominates

In [ ]:
texts = [d["text"] for d in SAMPLE_DOCS]
bm = BM25(texts)
scores = bm.scores("what does error E-4071 mean")
for i in np.argsort(scores)[::-1][:3]:
    print(round(scores[i], 2), SAMPLE_DOCS[i]["title"])

### Hybrid search with reciprocal rank fusion

Dense and keyword scores are on different scales, so do not add them. **Reciprocal rank fusion (RRF)** combines *rankings* instead: each document earns $1/(k + \text{rank})$ from each retriever, with $k \approx 60$, and the sums are re-sorted. Documents ranked well by both rise to the top.

In [ ]:
def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    scores = Counter()
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += 1 / (k + rank + 1)
    return [doc_id for doc_id, _ in scores.most_common()]

print(rrf([[0, 1, 2, 3], [1, 0, 3, 2], [2, 1, 0, 3]]))   # [1, 0, 2, 3]

### Reranking

First-stage retrieval is built for speed: query and document are embedded *separately*. A **cross-encoder reranker** reads the query and a candidate *together* and outputs a relevance score. It is far more accurate and far too slow to run on the whole collection, so use it on the top 20 to 50 candidates and keep the best 3 to 8.

*Shown for reference, not run here:*

```python
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(question, chunks[i]) for i in candidate_ids])
best = [candidate_ids[i] for i in np.argsort(-scores)[:5]]
```

## Grounded generation

### The prompt

Number the retrieved chunks, tell the model to answer only from them, to cite them, and to say when they do not contain the answer.

In [ ]:
RAG_PROMPT = """You answer questions about field instruments using only the sources below.

<sources>
{sources}
</sources>

Rules:
- Answer only from the sources. If they do not contain the answer, say so plainly and suggest
  what document might help. Do not use outside knowledge for facts.
- Cite sources inline as [1], [2] after the sentence they support.
- If sources disagree, say so and cite both.
- Be concise: a direct answer first, then any necessary detail or steps.

Question: {question}"""

def format_sources(hits: list[dict]) -> str:
    return "\n\n".join(
        f'<source id="{i}" title="{h["title"]}" section="{h["section"]}">\n{h["text"]}\n</source>'
        for i, h in enumerate(hits, start=1)
    )

### The whole pipeline, no framework

In [ ]:
# Downloads a model or data on first run
import numpy as np, anthropic
from sentence_transformers import SentenceTransformer, CrossEncoder

class MiniRAG:
    def __init__(self, embed_model="all-MiniLM-L6-v2",
                 rerank_model="cross-encoder/ms-marco-MiniLM-L-6-v2", llm_model="claude-sonnet-5-5"):
        self.embedder = SentenceTransformer(embed_model)
        self.reranker = CrossEncoder(rerank_model)
        self.client = anthropic.Anthropic()
        self.llm_model = llm_model
        self.chunks: list[dict] = []

    def index(self, documents: list[dict]):
        """documents: [{"title": ..., "text": ...}, ...]"""
        for doc in documents:
            for j, chunk in enumerate(chunk_text(doc["text"])):
                section = chunk.splitlines()[0][:80]               # crude: first line as section label
                self.chunks.append({"title": doc["title"], "section": section,
                                    "text": chunk, "id": f'{doc["title"]}#{j}'})
        texts = [f'{c["title"]}\n{c["text"]}' for c in self.chunks]   # title travels with the chunk
        self.vecs = self.embedder.encode(texts, normalize_embeddings=True)
        self.bm25 = BM25(texts)

    def retrieve(self, question: str, k_first=30, k_final=5) -> list[dict]:
        q = self.embedder.encode([question], normalize_embeddings=True)[0]
        dense = list(np.argsort(-(self.vecs @ q))[:k_first])
        sparse = list(np.argsort(-np.array(self.bm25.scores(question)))[:k_first])
        candidates = rrf([dense, sparse])[:k_first]
        scores = self.reranker.predict([(question, self.chunks[i]["text"]) for i in candidates])
        return [self.chunks[candidates[i]] for i in np.argsort(-scores)[:k_final]]

    def answer(self, question: str) -> dict:
        hits = self.retrieve(question)
        prompt = RAG_PROMPT.format(sources=format_sources(hits), question=question)
        r = self.client.messages.create(model=self.llm_model, max_tokens=700, temperature=0,
                                        messages=[{"role": "user", "content": prompt}])
        return {"answer": r.content[0].text, "sources": [h["id"] for h in hits]}

# rag = MiniRAG()
# rag.index([{"title": p.stem, "text": p.read_text()} for p in Path("docs").glob("*.md")])
# print(rag.answer("How often should the desiccant be replaced in humid sites?"))

In [ ]:
# Downloads a model or data on first run; Needs an API key (see the setup cell)
rag = MiniRAG()
rag.index(SAMPLE_DOCS)
for c in rag.retrieve("How often should the desiccant be replaced?", k_final=3):
    print(c["id"])
print(rag.answer("How often should the desiccant be replaced at a humid coastal site?")["answer"])

## Evaluate and improve

### Retrieval metrics

- **Recall@k**: fraction of questions whose relevant chunk appears in the top $k$. If the right chunk is not retrieved, the model cannot answer correctly; this is the first thing to measure.
- **MRR** (mean reciprocal rank): rewards getting the relevant chunk near the top.

In [ ]:
def retrieval_eval(rag, eval_set, k=5):
    hits, rr = 0, 0.0
    for item in eval_set:                                   # {"question": ..., "gold_id": "manual-x#12"}
        ids = [c["id"] for c in rag.retrieve(item["question"], k_final=k)]
        if item["gold_id"] in ids:
            hits += 1
            rr += 1 / (ids.index(item["gold_id"]) + 1)
    return {"recall@k": hits / len(eval_set), "mrr": rr / len(eval_set)}

In [ ]:
# Downloads a model or data on first run
eval_set = [
    {"question": "What should I do about error E-4071?", "gold_id": "Filter wheel#0"},
    {"question": "The tracker lost the sun, what do I check?", "gold_id": "Sun tracker and GPS#0"},
    {"question": "Why did logging stop after a failed upload?", "gold_id": "Data upload#0"},
    {"question": "My spectra got noisy in the heat", "gold_id": "Detector temperature and noise#0"},
]
print(retrieval_eval(rag, eval_set, k=3))

## RAG or long context?

### Hands-on: build, measure and improve a RAG system

1. Pick a document collection you care about: product manuals, a set of papers, a project's documentation, or the Markdown sources of your learning log. Aim for at least 50 pages.
2. Index it with `MiniRAG`. Write 25 evaluation questions with the chunk id that answers each, plus 5 questions the documents cannot answer.
3. Measure recall@5 and MRR for: dense only, BM25 only, hybrid, and hybrid plus reranker. Record a table.
4. Answer all 30 questions and grade faithfulness and abstention by hand. Find your three worst failures and classify each with the diagnosis table.
5. Fix one failure class and re-measure.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/13-rag.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.